# Experiment: exp_003_perch_downstream_reproduction

Objective:
- Reproduce the downstream part of the strongest Perch-style reference using only cached `perch_meta` outputs and local labels.
- Confirm whether the gain pattern remains `raw Perch scores < metadata-prior baseline < embedding probe`.
- Save notebook-native artifacts that we can compare later against our own soundscape finetuning run.


## Plan

- Hypothesis: most of the first gain should come from `site/hour/site-hour` priors plus texture-aware smoothing, and the probe should add a smaller but still meaningful second boost.
- Variables to sweep: frozen best fusion parameters, optional probe grid, and ablation switches for event-only vs texture-only priors.
- Metrics to record: macro ROC-AUC on the `59` fully labeled soundscape files, class coverage, number of modeled probe classes, and classwise deltas.
- Success criterion: reproduce the qualitative ranking of the reference stack and write compact result files under `experiments/outputs/exp_003_perch_downstream_reproduction/`.


In [ ]:
# Setup: imports, configuration, and reproducibility
from __future__ import annotations

import io
import json
import re
import tarfile
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
try:
    from tqdm.auto import tqdm
except ModuleNotFoundError:
    def tqdm(iterable=None, *args, **kwargs):
        return iterable if iterable is not None else []

warnings.filterwarnings("ignore")
SEED = 7
np.random.seed(SEED)

@dataclass
class Config:
    root: Path = Path.cwd()
    data_dir: Path = Path.cwd() / 'data' / 'birdclef-2026'
    perch_cache_dir: Path = Path.cwd() / 'data' / 'perch_meta'
    perch_model_archive: Path = Path.cwd() / 'data' / 'models' / 'bird-vocalization-classifier-tensorflow2-perch_v2_cpu-v1.tar.gz'
    output_dir: Path = Path.cwd() / 'experiments' / 'outputs' / 'exp_003_perch_downstream_reproduction'
    n_windows: int = 12
    best_fusion: dict = None
    frozen_best_probe: dict = None
    run_ablation_suite: bool = True
    run_probe_check: bool = True
    run_probe_grid: bool = False
    verbose: bool = True

    def __post_init__(self):
        if self.best_fusion is None:
            self.best_fusion = {
                'lambda_event': 0.4,
                'lambda_texture': 1.0,
                'lambda_proxy_texture': 0.8,
                'smooth_texture': 0.35,
            }
        if self.frozen_best_probe is None:
            self.frozen_best_probe = {
                'pca_dim': 64,
                'min_pos': 8,
                'C': 0.50,
                'alpha': 0.40,
            }

CFG = Config()
CFG.output_dir.mkdir(parents=True, exist_ok=True)

print(json.dumps({
    'seed': SEED,
    'data_dir': str(CFG.data_dir),
    'perch_cache_dir': str(CFG.perch_cache_dir),
    'perch_model_archive': str(CFG.perch_model_archive),
    'output_dir': str(CFG.output_dir),
    'run_probe_grid': CFG.run_probe_grid,
}, indent=2))


In [ ]:
# Lightweight helpers
FNAME_RE = re.compile(r"BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg")
TEXTURE_TAXA = {'Amphibia', 'Insecta'}


def parse_soundscape_labels(x):
    if pd.isna(x):
        return []
    return [token.strip() for token in str(x).split(';') if token.strip()]


def union_labels(series):
    labels = sorted(set(label for item in series for label in parse_soundscape_labels(item)))
    return labels


def parse_soundscape_filename(name):
    match = FNAME_RE.match(name)
    if not match:
        return {
            'file_id': None,
            'site': None,
            'date': pd.NaT,
            'time_utc': None,
            'hour_utc': -1,
            'month': -1,
        }
    file_id, site, ymd, hms = match.groups()
    dt = pd.to_datetime(ymd, format='%Y%m%d', errors='coerce')
    return {
        'file_id': file_id,
        'site': site,
        'date': dt,
        'time_utc': hms,
        'hour_utc': int(hms[:2]),
        'month': int(dt.month) if pd.notna(dt) else -1,
    }


def macro_auc_skip_empty(y_true, y_score):
    keep = y_true.sum(axis=0) > 0
    return float(roc_auc_score(y_true[:, keep], y_score[:, keep], average='macro'))


def classwise_auc_table(y_true, y_score, labels):
    rows = []
    for idx, label in enumerate(labels):
        y = y_true[:, idx]
        positives = int(y.sum())
        negatives = int(len(y) - positives)
        if positives == 0 or negatives == 0:
            auc = np.nan
        else:
            auc = float(roc_auc_score(y, y_score[:, idx]))
        rows.append({
            'primary_label': label,
            'positives': positives,
            'negatives': negatives,
            'auc': auc,
        })
    return pd.DataFrame(rows)


def smooth_cols_fixed12(scores, cols, alpha=0.35, n_windows=12):
    if alpha <= 0 or len(cols) == 0:
        return scores.copy()

    s = scores.copy()
    assert len(s) % n_windows == 0, f'Expected full-file blocks of {n_windows} windows'
    view = s.reshape(-1, n_windows, s.shape[1])

    x = view[:, :, cols]
    prev_x = np.concatenate([x[:, :1, :], x[:, :-1, :]], axis=1)
    next_x = np.concatenate([x[:, 1:, :], x[:, -1:, :]], axis=1)

    view[:, :, cols] = (1.0 - alpha) * x + 0.5 * alpha * (prev_x + next_x)
    return s


def seq_features_1d(v, n_windows=12):
    assert len(v) % n_windows == 0, f'Expected full-file blocks of {n_windows} windows'
    x = v.reshape(-1, n_windows)

    prev_v = np.concatenate([x[:, :1], x[:, :-1]], axis=1).reshape(-1)
    next_v = np.concatenate([x[:, 1:], x[:, -1:]], axis=1).reshape(-1)
    mean_v = np.repeat(x.mean(axis=1), n_windows)
    max_v = np.repeat(x.max(axis=1), n_windows)

    return prev_v, next_v, mean_v, max_v


In [ ]:
# Load soundscape truth, cached Perch outputs, and scientific-name mapping

taxonomy = pd.read_csv(CFG.data_dir / 'taxonomy.csv')
sample_sub = pd.read_csv(CFG.data_dir / 'sample_submission.csv')
soundscape_labels = pd.read_csv(CFG.data_dir / 'train_soundscapes_labels.csv')

PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES = len(PRIMARY_LABELS)
N_WINDOWS = int(CFG.n_windows)

label_to_idx = {label: idx for idx, label in enumerate(PRIMARY_LABELS)}

sc_clean = (
    soundscape_labels
    .groupby(['filename', 'start', 'end'])['primary_label']
    .apply(union_labels)
    .reset_index(name='label_list')
)
sc_clean['start_sec'] = pd.to_timedelta(sc_clean['start']).dt.total_seconds().astype(int)
sc_clean['end_sec'] = pd.to_timedelta(sc_clean['end']).dt.total_seconds().astype(int)
sc_clean['row_id'] = sc_clean['filename'].str.replace('.ogg', '', regex=False) + '_' + sc_clean['end_sec'].astype(str)
sc_meta = sc_clean['filename'].apply(parse_soundscape_filename).apply(pd.Series)
sc_clean = pd.concat([sc_clean, sc_meta], axis=1)

windows_per_file = sc_clean.groupby('filename').size()
full_files = sorted(windows_per_file[windows_per_file == N_WINDOWS].index.tolist())
sc_clean['file_fully_labeled'] = sc_clean['filename'].isin(full_files)

Y_SC = np.zeros((len(sc_clean), N_CLASSES), dtype=np.uint8)
for row_idx, labels in enumerate(sc_clean['label_list']):
    indices = [label_to_idx[label] for label in labels if label in label_to_idx]
    if indices:
        Y_SC[row_idx, indices] = 1

full_truth = (
    sc_clean[sc_clean['file_fully_labeled']]
    .sort_values(['filename', 'end_sec'])
    .reset_index(drop=False)
)
Y_FULL_TRUTH = Y_SC[full_truth['index'].to_numpy()]

meta_full = pd.read_parquet(CFG.perch_cache_dir / 'full_perch_meta.parquet')
perch_arrays = np.load(CFG.perch_cache_dir / 'full_perch_arrays.npz')
scores_full_raw = perch_arrays['scores_full_raw'].astype(np.float32)
emb_full = perch_arrays['emb_full'].astype(np.float32)

full_truth_aligned = full_truth.set_index('row_id').loc[meta_full['row_id']].reset_index()
Y_FULL = Y_SC[full_truth_aligned['index'].to_numpy()]

assert np.all(full_truth_aligned['filename'].values == meta_full['filename'].values)
assert np.all(full_truth_aligned['row_id'].values == meta_full['row_id'].values)
assert len(meta_full) == len(scores_full_raw) == len(emb_full) == len(Y_FULL)

with tarfile.open(CFG.perch_model_archive, 'r:gz') as tf:
    label_member = [member for member in tf.getmembers() if member.name.endswith('assets/labels.csv')][0]
    label_stream = tf.extractfile(label_member)
    bc_labels = (
        pd.read_csv(io.TextIOWrapper(label_stream, encoding='utf-8'))
        .reset_index()
        .rename(columns={'index': 'bc_index', 'inat2024_fsd50k': 'scientific_name'})
    )

NO_LABEL_INDEX = len(bc_labels)
MANUAL_SCIENTIFIC_NAME_MAP = {}

taxonomy = taxonomy.copy()
taxonomy['primary_label'] = taxonomy['primary_label'].astype(str)
taxonomy['scientific_name_lookup'] = taxonomy['scientific_name'].replace(MANUAL_SCIENTIFIC_NAME_MAP)

bc_lookup = bc_labels.rename(columns={'scientific_name': 'scientific_name_lookup'})
mapping = taxonomy.merge(
    bc_lookup[['scientific_name_lookup', 'bc_index']],
    on='scientific_name_lookup',
    how='left',
)
mapping['bc_index'] = mapping['bc_index'].fillna(NO_LABEL_INDEX).astype(int)

label_to_bc_index = mapping.set_index('primary_label')['bc_index']
BC_INDICES = np.array([int(label_to_bc_index.loc[label]) for label in PRIMARY_LABELS], dtype=np.int32)
MAPPED_MASK = BC_INDICES != NO_LABEL_INDEX
MAPPED_POS = np.where(MAPPED_MASK)[0].astype(np.int32)
UNMAPPED_POS = np.where(~MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_INDICES = BC_INDICES[MAPPED_MASK].astype(np.int32)

CLASS_NAME_MAP = taxonomy.set_index('primary_label')['class_name'].to_dict()
ACTIVE_CLASSES = [PRIMARY_LABELS[i] for i in np.where(Y_SC.sum(axis=0) > 0)[0]]

idx_active_texture = np.array(
    [label_to_idx[label] for label in ACTIVE_CLASSES if CLASS_NAME_MAP.get(label) in TEXTURE_TAXA],
    dtype=np.int32,
)
idx_active_event = np.array(
    [label_to_idx[label] for label in ACTIVE_CLASSES if CLASS_NAME_MAP.get(label) not in TEXTURE_TAXA],
    dtype=np.int32,
)

idx_mapped_active_texture = idx_active_texture[MAPPED_MASK[idx_active_texture]]
idx_mapped_active_event = idx_active_event[MAPPED_MASK[idx_active_event]]
idx_unmapped_active_texture = idx_active_texture[~MAPPED_MASK[idx_active_texture]]
idx_unmapped_active_event = idx_active_event[~MAPPED_MASK[idx_active_event]]
idx_unmapped_inactive = np.array(
    [i for i in UNMAPPED_POS if PRIMARY_LABELS[i] not in ACTIVE_CLASSES],
    dtype=np.int32,
)

unmapped_df = mapping[mapping['bc_index'] == NO_LABEL_INDEX].copy()
unmapped_non_sonotype = unmapped_df[
    ~unmapped_df['primary_label'].astype(str).str.contains('son', na=False)
].copy()

def get_genus_hits(scientific_name):
    genus = str(scientific_name).split()[0]
    hits = bc_labels[
        bc_labels['scientific_name'].astype(str).str.match(rf'^{re.escape(genus)}\s', na=False)
    ].copy()
    return genus, hits

proxy_map = {}
for _, row in unmapped_non_sonotype.iterrows():
    target = row['primary_label']
    genus, hits = get_genus_hits(row['scientific_name'])
    if len(hits) > 0:
        proxy_map[target] = {
            'target_scientific_name': row['scientific_name'],
            'genus': genus,
            'bc_indices': hits['bc_index'].astype(int).tolist(),
            'proxy_scientific_names': hits['scientific_name'].tolist(),
        }

SELECTED_PROXY_TARGETS = sorted([
    target for target in proxy_map.keys()
    if CLASS_NAME_MAP.get(target) == 'Amphibia'
])
selected_proxy_pos = np.array([label_to_idx[label] for label in SELECTED_PROXY_TARGETS], dtype=np.int32)
selected_proxy_pos_to_bc = {
    label_to_idx[target]: np.array(proxy_map[target]['bc_indices'], dtype=np.int32)
    for target in SELECTED_PROXY_TARGETS
}

idx_selected_proxy_active_texture = np.intersect1d(selected_proxy_pos, idx_active_texture)
idx_selected_prioronly_active_texture = np.setdiff1d(idx_unmapped_active_texture, selected_proxy_pos)
idx_selected_prioronly_active_event = np.setdiff1d(idx_unmapped_active_event, selected_proxy_pos)

summary = {
    'sc_clean_rows': int(len(sc_clean)),
    'full_files': int(len(full_files)),
    'trusted_windows': int(len(full_truth)),
    'active_classes_in_full_windows': int((Y_FULL.sum(axis=0) > 0).sum()),
    'mapped_classes': int(MAPPED_MASK.sum()),
    'unmapped_classes': int((~MAPPED_MASK).sum()),
    'active_texture_classes': int(len(idx_active_texture)),
    'selected_proxy_active_texture': int(len(idx_selected_proxy_active_texture)),
}
print(json.dumps(summary, indent=2))


In [ ]:
# Metadata priors, fusion, and honest OOF baseline helpers
BEST = CFG.best_fusion


def fit_prior_tables(prior_df, y_prior):
    prior_df = prior_df.reset_index(drop=True)
    global_p = y_prior.mean(axis=0).astype(np.float32)

    site_keys = sorted(prior_df['site'].dropna().astype(str).unique().tolist())
    site_to_i = {key: idx for idx, key in enumerate(site_keys)}
    site_n = np.zeros(len(site_keys), dtype=np.float32)
    site_p = np.zeros((len(site_keys), y_prior.shape[1]), dtype=np.float32)
    for site in site_keys:
        idx = site_to_i[site]
        mask = prior_df['site'].astype(str).values == site
        site_n[idx] = mask.sum()
        site_p[idx] = y_prior[mask].mean(axis=0)

    hour_keys = sorted(prior_df['hour_utc'].dropna().astype(int).unique().tolist())
    hour_to_i = {hour: idx for idx, hour in enumerate(hour_keys)}
    hour_n = np.zeros(len(hour_keys), dtype=np.float32)
    hour_p = np.zeros((len(hour_keys), y_prior.shape[1]), dtype=np.float32)
    for hour in hour_keys:
        idx = hour_to_i[hour]
        mask = prior_df['hour_utc'].astype(int).values == hour
        hour_n[idx] = mask.sum()
        hour_p[idx] = y_prior[mask].mean(axis=0)

    sh_to_i = {}
    sh_n_list = []
    sh_p_list = []
    for (site, hour), group_idx in prior_df.groupby(['site', 'hour_utc']).groups.items():
        sh_to_i[(str(site), int(hour))] = len(sh_n_list)
        group_idx = np.array(list(group_idx))
        sh_n_list.append(len(group_idx))
        sh_p_list.append(y_prior[group_idx].mean(axis=0))

    sh_n = np.array(sh_n_list, dtype=np.float32)
    sh_p = np.stack(sh_p_list).astype(np.float32) if len(sh_p_list) else np.zeros((0, y_prior.shape[1]), dtype=np.float32)

    return {
        'global_p': global_p,
        'site_to_i': site_to_i,
        'site_n': site_n,
        'site_p': site_p,
        'hour_to_i': hour_to_i,
        'hour_n': hour_n,
        'hour_p': hour_p,
        'sh_to_i': sh_to_i,
        'sh_n': sh_n,
        'sh_p': sh_p,
    }


def prior_logits_from_tables(sites, hours, tables, eps=1e-4):
    n_rows = len(sites)
    p = np.repeat(tables['global_p'][None, :], n_rows, axis=0).astype(np.float32, copy=True)

    site_idx = np.fromiter((tables['site_to_i'].get(str(site), -1) for site in sites), dtype=np.int32, count=n_rows)
    hour_idx = np.fromiter((tables['hour_to_i'].get(int(hour), -1) if int(hour) >= 0 else -1 for hour in hours), dtype=np.int32, count=n_rows)
    sh_idx = np.fromiter(
        (tables['sh_to_i'].get((str(site), int(hour)), -1) if int(hour) >= 0 else -1 for site, hour in zip(sites, hours)),
        dtype=np.int32,
        count=n_rows,
    )

    valid = hour_idx >= 0
    if valid.any():
        nh = tables['hour_n'][hour_idx[valid]][:, None]
        wh = nh / (nh + 8.0)
        p[valid] = wh * tables['hour_p'][hour_idx[valid]] + (1.0 - wh) * p[valid]

    valid = site_idx >= 0
    if valid.any():
        ns = tables['site_n'][site_idx[valid]][:, None]
        ws = ns / (ns + 8.0)
        p[valid] = ws * tables['site_p'][site_idx[valid]] + (1.0 - ws) * p[valid]

    valid = sh_idx >= 0
    if valid.any():
        nsh = tables['sh_n'][sh_idx[valid]][:, None]
        wsh = nsh / (nsh + 4.0)
        p[valid] = wsh * tables['sh_p'][sh_idx[valid]] + (1.0 - wsh) * p[valid]

    np.clip(p, eps, 1.0 - eps, out=p)
    return (np.log(p) - np.log1p(-p)).astype(np.float32, copy=False)


def fuse_scores_with_tables(
    base_scores,
    sites,
    hours,
    tables,
    lambda_event=BEST['lambda_event'],
    lambda_texture=BEST['lambda_texture'],
    lambda_proxy_texture=BEST['lambda_proxy_texture'],
    smooth_texture=BEST['smooth_texture'],
    suppress_unmapped_inactive=True,
):
    scores = base_scores.copy()
    prior = prior_logits_from_tables(sites, hours, tables)

    if len(idx_mapped_active_event):
        scores[:, idx_mapped_active_event] += lambda_event * prior[:, idx_mapped_active_event]
    if len(idx_mapped_active_texture):
        scores[:, idx_mapped_active_texture] += lambda_texture * prior[:, idx_mapped_active_texture]
    if len(idx_selected_proxy_active_texture):
        scores[:, idx_selected_proxy_active_texture] += lambda_proxy_texture * prior[:, idx_selected_proxy_active_texture]
    if len(idx_selected_prioronly_active_event):
        scores[:, idx_selected_prioronly_active_event] = lambda_event * prior[:, idx_selected_prioronly_active_event]
    if len(idx_selected_prioronly_active_texture):
        scores[:, idx_selected_prioronly_active_texture] = lambda_texture * prior[:, idx_selected_prioronly_active_texture]
    if suppress_unmapped_inactive and len(idx_unmapped_inactive):
        scores[:, idx_unmapped_inactive] = -8.0

    scores = smooth_cols_fixed12(scores, idx_active_texture, alpha=smooth_texture, n_windows=N_WINDOWS)
    return scores.astype(np.float32, copy=False), prior


def build_oof_variant(
    scores_raw,
    meta_df,
    sc_clean_df,
    y_sc,
    lambda_event,
    lambda_texture,
    lambda_proxy_texture,
    smooth_texture,
    suppress_unmapped_inactive=True,
    n_splits=5,
):
    groups = meta_df['filename'].to_numpy()
    gkf = GroupKFold(n_splits=n_splits)
    oof_scores = np.zeros_like(scores_raw, dtype=np.float32)
    oof_prior = np.zeros_like(scores_raw, dtype=np.float32)

    for _, (train_idx, valid_idx) in enumerate(gkf.split(scores_raw, groups=groups), 1):
        train_idx = np.sort(train_idx)
        valid_idx = np.sort(valid_idx)
        valid_files = set(meta_df.iloc[valid_idx]['filename'].tolist())

        prior_mask = ~sc_clean_df['filename'].isin(valid_files).values
        prior_df_fold = sc_clean_df.loc[prior_mask].reset_index(drop=True)
        y_prior_fold = y_sc[prior_mask]
        tables = fit_prior_tables(prior_df_fold, y_prior_fold)

        valid_scores, valid_prior = fuse_scores_with_tables(
            scores_raw[valid_idx],
            sites=meta_df.iloc[valid_idx]['site'].to_numpy(),
            hours=meta_df.iloc[valid_idx]['hour_utc'].to_numpy(),
            tables=tables,
            lambda_event=lambda_event,
            lambda_texture=lambda_texture,
            lambda_proxy_texture=lambda_proxy_texture,
            smooth_texture=smooth_texture,
            suppress_unmapped_inactive=suppress_unmapped_inactive,
        )
        oof_scores[valid_idx] = valid_scores
        oof_prior[valid_idx] = valid_prior

    return oof_scores, oof_prior


def run_ablation_suite():
    rows = []
    variants = [
        {
            'name': 'raw_scores',
            'lambda_event': 0.0,
            'lambda_texture': 0.0,
            'lambda_proxy_texture': 0.0,
            'smooth_texture': 0.0,
            'suppress_unmapped_inactive': False,
        },
        {
            'name': 'inactive_unmapped_suppression_only',
            'lambda_event': 0.0,
            'lambda_texture': 0.0,
            'lambda_proxy_texture': 0.0,
            'smooth_texture': 0.0,
            'suppress_unmapped_inactive': True,
        },
        {
            'name': 'event_priors_only',
            'lambda_event': BEST['lambda_event'],
            'lambda_texture': 0.0,
            'lambda_proxy_texture': 0.0,
            'smooth_texture': 0.0,
            'suppress_unmapped_inactive': True,
        },
        {
            'name': 'texture_priors_only',
            'lambda_event': 0.0,
            'lambda_texture': BEST['lambda_texture'],
            'lambda_proxy_texture': BEST['lambda_proxy_texture'],
            'smooth_texture': 0.0,
            'suppress_unmapped_inactive': True,
        },
        {
            'name': 'event_texture_priors',
            'lambda_event': BEST['lambda_event'],
            'lambda_texture': BEST['lambda_texture'],
            'lambda_proxy_texture': BEST['lambda_proxy_texture'],
            'smooth_texture': 0.0,
            'suppress_unmapped_inactive': True,
        },
        {
            'name': 'event_texture_priors_smooth',
            'lambda_event': BEST['lambda_event'],
            'lambda_texture': BEST['lambda_texture'],
            'lambda_proxy_texture': BEST['lambda_proxy_texture'],
            'smooth_texture': BEST['smooth_texture'],
            'suppress_unmapped_inactive': True,
        },
    ]

    for variant in variants:
        if variant['name'] == 'raw_scores':
            oof_scores = scores_full_raw.copy()
        else:
            oof_scores, _ = build_oof_variant(
                scores_raw=scores_full_raw,
                meta_df=meta_full,
                sc_clean_df=sc_clean,
                y_sc=Y_SC,
                lambda_event=variant['lambda_event'],
                lambda_texture=variant['lambda_texture'],
                lambda_proxy_texture=variant['lambda_proxy_texture'],
                smooth_texture=variant['smooth_texture'],
                suppress_unmapped_inactive=variant['suppress_unmapped_inactive'],
            )

        auc_value = macro_auc_skip_empty(Y_FULL, oof_scores)
        rows.append({**variant, 'macro_auc': auc_value})

    return pd.DataFrame(rows).sort_values('macro_auc', ascending=False).reset_index(drop=True)


In [ ]:
# Embedding-probe helpers
BEST_PROBE = CFG.frozen_best_probe


def build_class_features(emb_proj, raw_col, prior_col, base_col):
    prev_base, next_base, mean_base, max_base = seq_features_1d(base_col, n_windows=N_WINDOWS)
    features = np.concatenate([
        emb_proj,
        raw_col[:, None],
        prior_col[:, None],
        base_col[:, None],
        prev_base[:, None],
        next_base[:, None],
        mean_base[:, None],
        max_base[:, None],
    ], axis=1)
    return features.astype(np.float32, copy=False)


def run_oof_embedding_probe(
    scores_raw,
    emb,
    meta_df,
    y_true,
    sc_clean_df,
    y_sc,
    pca_dim=64,
    min_pos=8,
    C=0.25,
    alpha=0.5,
):
    groups = meta_df['filename'].to_numpy()
    gkf = GroupKFold(n_splits=5)

    oof_base_local = np.zeros_like(scores_raw, dtype=np.float32)
    oof_final = np.zeros_like(scores_raw, dtype=np.float32)
    modeled_counts = np.zeros(scores_raw.shape[1], dtype=np.int32)

    split_list = list(gkf.split(scores_raw, groups=groups))
    for fold, (train_idx, valid_idx) in enumerate(tqdm(split_list, desc='Embedding-probe folds', disable=not CFG.verbose), 1):
        train_idx = np.sort(train_idx)
        valid_idx = np.sort(valid_idx)
        valid_files = set(meta_df.iloc[valid_idx]['filename'].tolist())

        prior_mask = ~sc_clean_df['filename'].isin(valid_files).values
        prior_df_fold = sc_clean_df.loc[prior_mask].reset_index(drop=True)
        y_prior_fold = y_sc[prior_mask]
        tables = fit_prior_tables(prior_df_fold, y_prior_fold)

        base_tr, prior_tr = fuse_scores_with_tables(
            scores_raw[train_idx],
            sites=meta_df.iloc[train_idx]['site'].to_numpy(),
            hours=meta_df.iloc[train_idx]['hour_utc'].to_numpy(),
            tables=tables,
        )
        base_va, prior_va = fuse_scores_with_tables(
            scores_raw[valid_idx],
            sites=meta_df.iloc[valid_idx]['site'].to_numpy(),
            hours=meta_df.iloc[valid_idx]['hour_utc'].to_numpy(),
            tables=tables,
        )

        oof_base_local[valid_idx] = base_va
        oof_final[valid_idx] = base_va

        scaler = StandardScaler()
        emb_tr_scaled = scaler.fit_transform(emb[train_idx])
        emb_va_scaled = scaler.transform(emb[valid_idx])

        n_comp = min(pca_dim, emb_tr_scaled.shape[0] - 1, emb_tr_scaled.shape[1])
        pca = PCA(n_components=n_comp)
        z_tr = pca.fit_transform(emb_tr_scaled).astype(np.float32)
        z_va = pca.transform(emb_va_scaled).astype(np.float32)

        class_iterator = np.where(y_true[train_idx].sum(axis=0) >= min_pos)[0].tolist()
        for cls_idx in tqdm(class_iterator, desc=f'Fold {fold} classes', leave=False, disable=not CFG.verbose):
            y_train_cls = y_true[train_idx, cls_idx]
            if y_train_cls.sum() == 0 or y_train_cls.sum() == len(y_train_cls):
                continue

            x_train = build_class_features(
                z_tr,
                raw_col=scores_raw[train_idx, cls_idx],
                prior_col=prior_tr[:, cls_idx],
                base_col=base_tr[:, cls_idx],
            )
            x_valid = build_class_features(
                z_va,
                raw_col=scores_raw[valid_idx, cls_idx],
                prior_col=prior_va[:, cls_idx],
                base_col=base_va[:, cls_idx],
            )

            clf = LogisticRegression(
                C=C,
                max_iter=400,
                solver='liblinear',
                class_weight='balanced',
            )
            clf.fit(x_train, y_train_cls)
            pred_valid = clf.decision_function(x_valid).astype(np.float32)

            oof_final[valid_idx, cls_idx] = (1.0 - alpha) * base_va[:, cls_idx] + alpha * pred_valid
            modeled_counts[cls_idx] += 1

    return {
        'oof_base': oof_base_local,
        'oof_final': oof_final,
        'modeled_counts': modeled_counts,
        'score_base': macro_auc_skip_empty(y_true, oof_base_local),
        'score_final': macro_auc_skip_empty(y_true, oof_final),
    }


In [ ]:
# Run the downstream reproduction and save artifacts
ablation_results = None
if CFG.run_ablation_suite:
    ablation_results = run_ablation_suite()
    ablation_results.to_csv(CFG.output_dir / 'ablation_results.csv', index=False)
    print(ablation_results)

oof_base, oof_prior = build_oof_variant(
    scores_raw=scores_full_raw,
    meta_df=meta_full,
    sc_clean_df=sc_clean,
    y_sc=Y_SC,
    lambda_event=BEST['lambda_event'],
    lambda_texture=BEST['lambda_texture'],
    lambda_proxy_texture=BEST['lambda_proxy_texture'],
    smooth_texture=BEST['smooth_texture'],
    suppress_unmapped_inactive=True,
)
np.savez_compressed(
    CFG.output_dir / 'full_oof_meta_features.npz',
    oof_base=oof_base,
    oof_prior=oof_prior,
)

raw_local_auc = macro_auc_skip_empty(Y_FULL, scores_full_raw)
baseline_oof_auc = macro_auc_skip_empty(Y_FULL, oof_base)

probe_result = None
if CFG.run_probe_check:
    probe_result = run_oof_embedding_probe(
        scores_raw=scores_full_raw,
        emb=emb_full,
        meta_df=meta_full,
        y_true=Y_FULL,
        sc_clean_df=sc_clean,
        y_sc=Y_SC,
        pca_dim=int(BEST_PROBE['pca_dim']),
        min_pos=int(BEST_PROBE['min_pos']),
        C=float(BEST_PROBE['C']),
        alpha=float(BEST_PROBE['alpha']),
    )
    np.savez_compressed(
        CFG.output_dir / 'probe_oof_outputs.npz',
        oof_base=probe_result['oof_base'],
        oof_final=probe_result['oof_final'],
        modeled_counts=probe_result['modeled_counts'],
    )

probe_grid_results = None
if CFG.run_probe_grid:
    param_grid = [
        {'pca_dim': 32, 'min_pos': 8, 'C': 0.25, 'alpha': 0.4},
        {'pca_dim': 64, 'min_pos': 8, 'C': 0.25, 'alpha': 0.4},
        {'pca_dim': 64, 'min_pos': 8, 'C': 0.25, 'alpha': 0.5},
        {'pca_dim': 64, 'min_pos': 12, 'C': 0.25, 'alpha': 0.4},
        {'pca_dim': 96, 'min_pos': 8, 'C': 0.25, 'alpha': 0.4},
        {'pca_dim': 64, 'min_pos': 8, 'C': 0.50, 'alpha': 0.4},
    ]
    rows = []
    for params in tqdm(param_grid, desc='Probe grid', disable=not CFG.verbose):
        grid_out = run_oof_embedding_probe(
            scores_raw=scores_full_raw,
            emb=emb_full,
            meta_df=meta_full,
            y_true=Y_FULL,
            sc_clean_df=sc_clean,
            y_sc=Y_SC,
            pca_dim=params['pca_dim'],
            min_pos=params['min_pos'],
            C=params['C'],
            alpha=params['alpha'],
        )
        rows.append({
            **params,
            'baseline_oof_auc': grid_out['score_base'],
            'probe_oof_auc': grid_out['score_final'],
            'delta': grid_out['score_final'] - grid_out['score_base'],
            'n_modeled_classes': int((grid_out['modeled_counts'] > 0).sum()),
        })
    probe_grid_results = pd.DataFrame(rows).sort_values('probe_oof_auc', ascending=False).reset_index(drop=True)
    probe_grid_results.to_csv(CFG.output_dir / 'probe_grid_results.csv', index=False)

classwise_raw = classwise_auc_table(Y_FULL, scores_full_raw, PRIMARY_LABELS).rename(columns={'auc': 'raw_auc'})
classwise_base = classwise_auc_table(Y_FULL, oof_base, PRIMARY_LABELS).rename(columns={'auc': 'baseline_auc'})
classwise_compare = classwise_raw[['primary_label', 'positives', 'negatives', 'raw_auc']].merge(
    classwise_base[['primary_label', 'baseline_auc']],
    on='primary_label',
    how='left',
)
if probe_result is not None:
    classwise_probe = classwise_auc_table(Y_FULL, probe_result['oof_final'], PRIMARY_LABELS).rename(columns={'auc': 'probe_auc'})
    classwise_compare = classwise_compare.merge(classwise_probe[['primary_label', 'probe_auc']], on='primary_label', how='left')
    classwise_compare['baseline_delta_vs_raw'] = classwise_compare['baseline_auc'] - classwise_compare['raw_auc']
    classwise_compare['probe_delta_vs_base'] = classwise_compare['probe_auc'] - classwise_compare['baseline_auc']
classwise_compare.to_csv(CFG.output_dir / 'classwise_auc_comparison.csv', index=False)

result_snapshot = {
    'experiment_id': 'exp_003',
    'experiment_name': 'perch_downstream_reproduction',
    'seed': SEED,
    'trusted_files': int(len(full_files)),
    'trusted_windows': int(len(meta_full)),
    'raw_local_auc': raw_local_auc,
    'baseline_oof_auc': baseline_oof_auc,
    'probe_oof_auc': None if probe_result is None else float(probe_result['score_final']),
    'probe_delta': None if probe_result is None else float(probe_result['score_final'] - probe_result['score_base']),
    'modeled_probe_classes': None if probe_result is None else int((probe_result['modeled_counts'] > 0).sum()),
    'best_fusion': BEST,
    'best_probe': BEST_PROBE,
}
(CFG.output_dir / 'result_snapshot.json').write_text(json.dumps(result_snapshot, indent=2))

print(json.dumps(result_snapshot, indent=2))
if probe_result is not None:
    top_probe_delta = (
        classwise_compare.dropna(subset=['probe_delta_vs_base'])
        .sort_values('probe_delta_vs_base', ascending=False)
        .head(15)
        [['primary_label', 'positives', 'baseline_auc', 'probe_auc', 'probe_delta_vs_base']]
    )
    print('\nTop probe gains:')
    print(top_probe_delta.to_string(index=False))


## Results

- Run the experiment cell above to populate `ablation_results.csv`, `classwise_auc_comparison.csv`, `full_oof_meta_features.npz`, and `result_snapshot.json`.
- The key comparison to watch is `raw_local_auc` vs `baseline_oof_auc` vs `probe_oof_auc`.
- If texture-only priors again explain most of the uplift, that is a strong argument for bringing texture-aware handling into the native pipeline before large architecture changes.


In [ ]:
# Load saved artifacts for a compact review pass
snapshot_path = CFG.output_dir / 'result_snapshot.json'
ablation_path = CFG.output_dir / 'ablation_results.csv'
classwise_path = CFG.output_dir / 'classwise_auc_comparison.csv'

if snapshot_path.exists():
    snapshot = json.loads(snapshot_path.read_text())
    print('Snapshot:')
    print(json.dumps(snapshot, indent=2))
else:
    print('Run the experiment cell first to create result_snapshot.json')

if ablation_path.exists():
    print('\nAblations:')
    print(pd.read_csv(ablation_path).to_string(index=False))

if classwise_path.exists():
    compare_df = pd.read_csv(classwise_path)
    review_cols = ['primary_label', 'positives', 'raw_auc', 'baseline_auc']
    if 'probe_auc' in compare_df.columns:
        review_cols.extend(['probe_auc', 'probe_delta_vs_base'])
    print('\nClasswise head:')
    print(compare_df[review_cols].head(20).to_string(index=False))


## Next Steps

- Freeze the reproduced `Perch + priors + probe` results in the experiment table and notes.
- Start `exp_004_soundscape_finetuning` from our native `exp_002` checkpoint.
- Compare `exp_004` against this Perch downstream notebook on both local soundscape metrics and public Kaggle behavior.
